In [ ]:
import os, sys, json, shutil, numpy as np, subprocess, time
from pathlib import Path

COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))

SRC = Path('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/src')
sys.path.insert(0, str(SRC))

bp_members = Path('/kaggle/input/datasets/busyaprime/33-qcc-v8-members')
sub_dir = WORK / 'submission'
sub_dir.mkdir(exist_ok=True)
(sub_dir / 'dfthcsos').mkdir(exist_ok=True)

shutil.copy2(bp_members / 'invsqrt.kmb', sub_dir / 'invsqrt.kmb')
shutil.copy2(bp_members / 'dqi.kmb', sub_dir / 'dqi.kmb')
for p in bp_members.glob('dfthcsos__*.kmb'):
    shutil.copy2(p, sub_dir / 'dfthcsos' / p.name.replace('dfthcsos__', ''))
shutil.copy2(bp_members / 'dfthcsos__submission.npz', sub_dir / 'dfthcsos' / 'submission.npz')

print('Setup complete')


In [ ]:
# Run gauge.py with 10 sweeps, seed 0
IN = sub_dir / 'dfthcsos' / 'submission.npz'
OUT = WORK / 'gauged.npz'

t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'gauge.py'), str(IN), str(OUT), '--sweeps', '10'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC))
)
print('STDOUT:', result.stdout)
print('STDERR:', result.stderr[-300:] if result.stderr else 'NONE')
print(f'Time: {time.time()-t0:.0f}s')
print('Output exists:', OUT.exists())


In [ ]:
if OUT.exists():
    shutil.copy2(OUT, sub_dir / 'dfthcsos' / 'submission.npz')
    from dfthcsos.verify import verify as CV
    r = CV(sub_dir / 'dfthcsos')
    print('Full verify:', r['status'], r['score'])
    
    zip_path = WORK / 'submission.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for p in sorted(sub_dir.rglob('*')):
            if p.is_file():
                z.write(p, str(p.relative_to(sub_dir)))
    print('Wrote', zip_path, zip_path.stat().st_size, 'bytes')
else:
    print('Gauge failed')
